## use the CIFAR10 datasets to train a clasification model (need cpu to run), and the datasets is not match the nerual network.
## this is a simple example to show how to use the datasets to train a model.

### 1-import packages

In [20]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

### 2-define datasets & transforms & loaders

In [21]:
transfroms = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(5),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

tr_sets = datasets.MNIST(root='../data', train=True, download=True, transform=transfroms)
tr_loader = DataLoader(tr_sets, batch_size=32, shuffle=True)

### 3-train model

In [22]:
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

# use pre-trained model
# model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
model = models.resnet18(pretrained=True)

# 
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10)

d:\ML\Anaconda\envs\venv_pytorch\Lib\site-packages\torchvision\models\_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\ML\Anaconda\envs\venv_pytorch\Lib\site-packages\torchvision\models\_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


### 4-define loss function and optimizer

In [23]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

### 5-start training

In [24]:
for epoch in range(10):
    for img, l in tr_loader:
        outputs = model(img)
        loss = criterion(outputs, l)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

KeyboardInterrupt: 

### 6-model import and deploy

In [ ]:
model.eval()
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# model.to(device)

# dmy_input = torch.randn(1, 1, 28, 28).to(device)
dmy_input = torch.randn(1, 1, 28, 28)

with torch.no_grad():
    torch.onnx.export(
        model, dmy_input, 
        'model.onnx', 
        input_names=['input'], 
        output_names=['output'],
        opset_version=12)